# 02 · Inspección
Primera revisión de la temporada 2025-26: forma, tipos, nulos y granularidad.

**Pregunta de este notebook:** ¿puedo confiar en esta tabla y entiendo qué representa cada fila?
Es el primer paso para responder: ¿qué diferencia a los equipos que ganan de los que pierden?

In [2]:
import pandas as pd

df = pd.read_parquet("../data/raw/team_game_log_2025_26.parquet")
df.sample(5, random_state=1)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
1268,22025,1610612744,GSW,Golden State Warriors,0022500612,2026-01-19,GSW vs. MIA,W,240,49,...,38,46,38,11,3,14,17,135,23,1
2270,22025,1610612741,CHI,Chicago Bulls,0022501109,2026-04-01,CHI vs. IND,L,240,47,...,32,46,22,6,4,20,14,126,-19,1
1617,22025,1610612752,NYK,New York Knicks,0022500780,2026-02-11,NYK @ PHI,W,240,48,...,39,51,41,13,6,15,16,138,49,1
499,22025,1610612766,CHA,Charlotte Hornets,0022500276,2025-11-23,CHA @ ATL,L,240,39,...,34,43,26,4,2,16,15,110,-3,1
535,22025,1610612756,PHX,Phoenix Suns,0022500068,2025-11-26,PHX @ SAC,W,240,42,...,30,47,25,14,5,17,14,112,12,1


A primera vista, cada fila corresponde a un equipo en un partido. La tabla tiene tres identificadores con propósitos distintos: SEASON_ID (la temporada), TEAM_ID (el equipo) y GAME_ID (el partido). Es una tabla plana o desnormalizada: el nombre y la sigla del equipo se repiten en cada uno de sus partidos, lo que facilita el análisis porque no hay que unir tablas.

Además de la fecha (GAME_DATE), cada fila trae las estadísticas del equipo en ese partido (tiros, rebotes, asistencias y puntos) y PLUS_MINUS, que para un equipo es la diferencia final de puntos. MATCHUP indica si jugó de local (vs.) o de visita (@), y MIN cuántos minutos jugó. VIDEO_AVAILABLE no aporta a las preguntas de este análisis.

## 2. ¿De qué tipo es cada columna?
revisaremos que tipo de datos tiene cada columna para verificar si necesitaremos alguna transformacion, fecha puede venir en string y no en date, o los porcentajes de tiro pueden venir en otro fomato que no sea en float64 con esto lo corroboramos 

In [4]:
df.dtypes

SEASON_ID                str
TEAM_ID                int64
TEAM_ABBREVIATION        str
TEAM_NAME                str
GAME_ID                  str
GAME_DATE                str
MATCHUP                  str
WL                       str
MIN                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PTS                    int64
PLUS_MINUS             int64
VIDEO_AVAILABLE        int64
dtype: object

Los tipos calzan con el contenido de cada columna:

Texto (str): los identificadores SEASON_ID y GAME_ID, la sigla y el nombre del equipo, MATCHUP, WL y GAME_DATE.
Enteros (int64): las estadísticas que se cuentan (tiros, rebotes, asistencias, puntos), MIN y PLUS_MINUS.
Decimales (float64): los tres porcentajes de tiro, escritos como fracción entre 0 y 1.

SEASON_ID y GAME_ID están bien como texto: son códigos, no cantidades. TEAM_ID también es un identificador aunque venga como número; no causa problemas, pero no se usa en cálculos.

El único tipo que hay que corregir es el de GAME_DATE: como texto, no permite filtrar por mes ni ordenar por calendario.

In [6]:
df.isna().sum()

SEASON_ID            0
TEAM_ID              0
TEAM_ABBREVIATION    0
TEAM_NAME            0
GAME_ID              0
GAME_DATE            0
MATCHUP              0
WL                   0
MIN                  0
FGM                  0
FGA                  0
FG_PCT               0
FG3M                 0
FG3A                 0
FG3_PCT              0
FTM                  0
FTA                  0
FT_PCT               0
OREB                 0
DREB                 0
REB                  0
AST                  0
STL                  0
BLK                  0
TOV                  0
PF                   0
PTS                  0
PLUS_MINUS           0
VIDEO_AVAILABLE      0
dtype: int64

Decisiones para la limpieza
Convertir GAME_DATE a fecha.
Crear la columna gano: 1 si WL es W y 0 si es L.
No usar VIDEO_AVAILABLE.